# Module 5: RAG Pipeline with Azure DocumentDB

**Time**: ~60 min  
**Environment**: Jupyter notebook in VS Code

This notebook is fully runnable. Enter your Azure DocumentDB connection string and OpenAI API key in Step 0, then run each cell in order. The notebook creates embeddings for RAG chunks, stores them in Azure DocumentDB, retrieves context with vector and hybrid search, and builds a grounded prompt for a chat model.

The final step prints the prompt your application would send to a chat model.


## Step 0: Connect and configure embeddings

This cell restores the MongoDB driver, accepts the DocumentDB connection string and OpenAI API key, and creates an embedding helper.

In [ ]:
#r "nuget: MongoDB.Driver, 3.4.0"
using MongoDB.Bson;
using MongoDB.Driver;
using System.Linq;
using System.Net.Http.Headers;
using System.Text;
using System.Text.Json;

var connectionString = Environment.GetEnvironmentVariable("DOCUMENTDB_CONNECTION_STRING") ?? "<paste-your-azure-documentdb-connection-string-here>";
var openAiApiKey = Environment.GetEnvironmentVariable("OPENAI_API_KEY") ?? "<paste-your-openai-api-key-here>";
var embeddingModel = Environment.GetEnvironmentVariable("OPENAI_EMBEDDING_MODEL") ?? "text-embedding-3-small";
if (connectionString.Contains("<paste")) throw new Exception("Paste your Azure DocumentDB connection string in this cell or set DOCUMENTDB_CONNECTION_STRING.");
if (openAiApiKey.Contains("<paste")) throw new Exception("Paste your OpenAI API key in this cell or set OPENAI_API_KEY.");
var client = new MongoClient(connectionString);
var db = client.GetDatabase("docdbworkshop");
var chunks = db.GetCollection<BsonDocument>("rag_chunks");
var http = new HttpClient();
db.RunCommand<BsonDocument>(new BsonDocument("ping", 1))

## Step 1: Generate embeddings for RAG chunks

Each chunk is embedded with OpenAI and stored in Azure DocumentDB.

In [ ]:
async Task<BsonArray> CreateEmbeddingAsync(string text)
{
    using var request = new HttpRequestMessage(HttpMethod.Post, "https://api.openai.com/v1/embeddings");
    request.Headers.Authorization = new AuthenticationHeaderValue("Bearer", openAiApiKey);
    request.Content = new StringContent(JsonSerializer.Serialize(new { model = embeddingModel, input = text }), Encoding.UTF8, "application/json");
    using var response = await http.SendAsync(request);
    var body = await response.Content.ReadAsStringAsync();
    response.EnsureSuccessStatusCode();
    using var json = JsonDocument.Parse(body);
    return new BsonArray(json.RootElement.GetProperty("data")[0].GetProperty("embedding").EnumerateArray().Select(v => v.GetDouble()));
}

var sourceChunks = new[] {
    new { Id="rag-001", Title="Vector search", Chunk="Azure DocumentDB vector search uses the $search stage with the cosmosSearch operator to retrieve documents by embedding similarity.", Url="module-4-search" },
    new { Id="rag-002", Title="Full-text search", Chunk="Azure DocumentDB full-text search uses createSearchIndexes and the $search text operator to return BM25-ranked keyword matches.", Url="module-4-search" },
    new { Id="rag-003", Title="Hybrid search", Chunk="Hybrid search runs BM25 keyword retrieval and vector retrieval, then combines ranked lists with Reciprocal Rank Fusion.", Url="module-4-search" },
    new { Id="rag-004", Title="Grounded generation", Chunk="A RAG pipeline retrieves relevant chunks from Azure DocumentDB and includes them in the model prompt so the answer is grounded in current application data.", Url="module-5-rag" }
};
db.DropCollection("rag_chunks");
chunks = db.GetCollection<BsonDocument>("rag_chunks");
var chunkDocs = new List<BsonDocument>();
foreach (var item in sourceChunks)
{
    chunkDocs.Add(new BsonDocument { {"_id", item.Id}, {"title", item.Title}, {"chunk", item.Chunk}, {"url", item.Url}, {"embedding", await CreateEmbeddingAsync(item.Chunk)} });
}
chunks.InsertMany(chunkDocs);
var embeddingDimensions = chunkDocs[0]["embedding"].AsBsonArray.Count;
new { Loaded = chunks.CountDocuments(FilterDefinition<BsonDocument>.Empty), EmbeddingDimensions = embeddingDimensions }

## Step 2: Create retrieval indexes

Create the vector and BM25 indexes using the actual embedding dimension count.

In [ ]:
db.RunCommand<BsonDocument>(new BsonDocument{{"createIndexes","rag_chunks"},{"indexes",new BsonArray{new BsonDocument{{"name","idx_chunk_embedding_diskann"},{"key",new BsonDocument("embedding","cosmosSearch")},{"cosmosSearchOptions",new BsonDocument{{"kind","vector-diskann"},{"dimensions",embeddingDimensions},{"similarity","COS"},{"maxDegree",32},{"lBuild",64}}}}}}});
db.RunCommand<BsonDocument>(new BsonDocument{{"createSearchIndexes","rag_chunks"},{"indexes",new BsonArray{new BsonDocument{{"name","idx_chunk_fts"},{"definition",new BsonDocument("mappings",new BsonDocument{{"dynamic",false},{"fields",new BsonDocument("chunk",new BsonDocument("type","string"))}})}}}}});

## Step 3: Generate a question embedding and retrieve context

The same OpenAI embedding model turns the user question into the query vector.

In [ ]:
var question = "How does DocumentDB retrieve context for RAG?";
var questionVector = await CreateEmbeddingAsync(question);
var vectorContext = chunks.Aggregate<BsonDocument>(new[] { new BsonDocument("$search", new BsonDocument("cosmosSearch", new BsonDocument{{"path","embedding"},{"vector",questionVector},{"k",3}})), new BsonDocument("$project", new BsonDocument{{"_id",1},{"title",1},{"chunk",1},{"url",1},{"score",new BsonDocument("$meta","searchScore")}}) }).ToList();
vectorContext

## Step 4: Build the grounded prompt

The retrieved chunks become the context block for the final chat-model prompt.

In [ ]:
var contextBlock = string.Join("

", vectorContext.Select((d, i) => $"[{i + 1}] {d["title"]}
{d["chunk"]}
Source: {d["url"]}"));
var groundedPrompt = $"""You are a helpful assistant for an Azure DocumentDB workshop.
Answer using only the context below. If the answer is missing, say you do not know.

<context>
{contextBlock}
</context>

Question: {question}""";
groundedPrompt